# Olist Data Overview

## Цель

Познакомиться со структурой исходных данных:
- определить размер таблиц;
- изучить столбцы и типы данных;
- проверить пропуски;
- проверить дубликаты;
- определить ключи;
- определить гранулярность таблиц;
- понять связи между таблицами.

In [1]:
import pandas as pd
print("pandas:", pd.__version__)

pandas: 3.0.6


In [2]:
DATA_PATH = "../data/raw/"
customers = pd.read_csv(DATA_PATH + "olist_customers_dataset.csv")
geolocation = pd.read_csv(DATA_PATH + "olist_geolocation_dataset.csv")
order_items = pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_PATH + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_PATH + "olist_order_reviews_dataset.csv")
orders = pd.read_csv(DATA_PATH + "olist_orders_dataset.csv")
products = pd.read_csv(DATA_PATH + "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_PATH + "olist_sellers_dataset.csv")
categories = pd.read_csv(
    DATA_PATH + "product_category_name_translation.csv"
)

In [3]:
tables = {
    "orders": orders,
    "customers": customers,
    "order_items": order_items,
    "products": products,
    "sellers": sellers,
    "payments": payments,
    "reviews": reviews,
    "categories": categories,
    "geolocation": geolocation
}

overview = pd.DataFrame({
    "rows": [df.shape[0] for df in tables.values()],
    "columns": [df.shape[1] for df in tables.values()]
}, index=tables.keys())

overview.index.name = "table"
overview

,rows,columns
table,,
orders,99441,8
customers,99441,5
order_items,112650,7
products,32951,9
sellers,3095,4
payments,103886,5
reviews,99224,7
categories,71,2
geolocation,1000163,5


## 1. Orders

Таблица с информацией о заказах.

In [4]:
# Первые строки таблицы
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
# Общая информация
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [6]:
# Проверка primary key
print("order_id уникален:", orders["order_id"].is_unique)
print("Пропусков в order_id:", orders["order_id"].isna().sum())

order_id уникален: True
Пропусков в order_id: 0


In [7]:
# Проверка полных дубликатов
print("Полных дубликатов:", orders.duplicated().sum())

Полных дубликатов: 0


In [8]:
# Распределение заказов по статусам
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [9]:
# Количество и доля пропусков по столбцам
missing = pd.DataFrame({
    "missing_count": orders.isna().sum(),
    "missing_percent": orders.isna().mean().mul(100).round(2)
})

missing

,missing_count,missing_percent
order_id,0,0.00
customer_id,0,0.00
order_status,0,0.00
order_purchase_timestamp,0,0.00
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98
order_estimated_delivery_date,0,0.00


### Вывод по `orders`

- Количество строк: **99 441**.
- Количество столбцов: **8**.
- Одна строка соответствует **одному заказу**.
- `order_id` является **уникальным идентификатором заказа**: все значения уникальны, пропуски отсутствуют. Поэтому столбец можно использовать как `PRIMARY KEY`.
- Полных дубликатов строк **нет**.
- В таблице присутствуют следующие статусы заказов:
    - `delivered` — 96 478;
    - `shipped` — 1 107;
    - `canceled` — 625;
    - `unavailable` — 609;
    - `invoiced` — 314;
    - `processing` — 301;
    - `created` — 5;
    - `approved` — 2.
- Большинство заказов имеют статус `delivered`.
- Пропуски присутствуют только в столбцах, связанных с этапами обработки и доставки заказа:
    - `order_approved_at` — 160 пропусков (0,16%);
    - `order_delivered_carrier_date` — 1 783 пропуска (1,79%);
    - `order_delivered_customer_date` — 2 965 пропусков (2,98%).
- В `order_id`, `customer_id`, `order_status`, `order_purchase_timestamp` и `order_estimated_delivery_date` пропусков нет.
- Столбцы с датами при первоначальной загрузке из CSV были прочитаны как строки (`str`), поэтому перед дальнейшим анализом их необходимо преобразовать в тип `datetime`.

## 2. Customers

Таблица с информацией о клиентах.

In [10]:
# Первые строки таблицы
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [11]:
# Общая информация
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [12]:
# Проверка primary key
print("customer_id уникален:", customers["customer_id"].is_unique)
print("Пропусков в customer_id:", customers["customer_id"].isna().sum())

customer_id уникален: True
Пропусков в customer_id: 0


In [13]:
# Проверка полных дубликатов
print("Полных дубликатов:", customers.duplicated().sum())

Полных дубликатов: 0


In [14]:
# Проверка идентификатора уникального покупателя
print(
    "customer_unique_id уникален:",
    customers["customer_unique_id"].is_unique
)

print(
    "Количество уникальных покупателей:",
    customers["customer_unique_id"].nunique()
)

customer_unique_id уникален: False
Количество уникальных покупателей: 96096


In [15]:
# Анализ повторяющихся customer_unique_id
customer_counts = customers["customer_unique_id"].value_counts()

print(
    "Покупателей с несколькими записями:",
    (customer_counts > 1).sum()
)

print(
    "Максимальное количество записей на одного покупателя:",
    customer_counts.max()
)

Покупателей с несколькими записями: 2997
Максимальное количество записей на одного покупателя: 17


### Вывод по `customers`

- Количество строк: **99 441**.
- Количество столбцов: **5**.
- Одна строка соответствует **записи клиента, связанной с отдельным заказом**.
- `customer_id` является уникальным идентификатором записи клиента: значения не повторяются и не содержат пропусков. Поэтому столбец можно использовать как `PRIMARY KEY`.
- Полных дубликатов строк **нет**.
- Пропуски во всех столбцах **отсутствуют**.
- `customer_unique_id` не является уникальным: среди 99 441 записей содержится **96 096 уникальных покупателей**.
- Один и тот же `customer_unique_id` может соответствовать нескольким `customer_id`, поскольку для разных заказов одного покупателя используются разные `customer_id`.
- У **2 997 покупателей** `customer_unique_id` встречается более одного раза.
- Максимальное количество записей, соответствующих одному `customer_unique_id`, составляет **17**.
- Поэтому `customer_id` следует использовать для связи клиента с конкретным заказом, а `customer_unique_id` — для подсчёта уникальных покупателей и анализа повторных покупок.
- Таблица также содержит географические характеристики клиента:
  - `customer_zip_code_prefix` — префикс почтового индекса;
  - `customer_city` — город;
  - `customer_state` — штат.
- `customer_zip_code_prefix` имеет тип `int64`, однако является идентификатором почтового индекса, а не числовой величиной.

## 3. Order Items

Таблица с информацией о товарных позициях, входящих в заказы.

In [16]:
# Первые строки таблицы
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [17]:
# Общая информация
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [18]:
# Проверка идентификаторов
print("order_id уникален:", order_items["order_id"].is_unique)
print("Пропусков в order_id:", order_items["order_id"].isna().sum())
print("order_item_id уникален:", order_items["order_item_id"].is_unique)

order_id уникален: False
Пропусков в order_id: 0
order_item_id уникален: False


In [19]:
# Проверка составного ключа
print(
    "Пара order_id + order_item_id уникальна:",
    ~order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).any()
)

Пара order_id + order_item_id уникальна: True


In [20]:
# Проверка полных дубликатов
print("Полных дубликатов:", order_items.duplicated().sum())

Полных дубликатов: 0


In [21]:
# Количество товарных позиций в заказах
items_per_order = order_items.groupby("order_id").size()

print("Уникальных заказов:", order_items["order_id"].nunique())
print("Заказов с несколькими позициями:", (items_per_order > 1).sum())
print("Максимальное количество позиций в заказе:", items_per_order.max())

Уникальных заказов: 98666
Заказов с несколькими позициями: 9803
Максимальное количество позиций в заказе: 21


### Вывод по `order_items`

- Количество строк: **112 650**.
- Количество столбцов: **7**.
- Одна строка соответствует **отдельной товарной позиции внутри заказа**.
- `order_id` не является уникальным, поскольку один заказ может содержать несколько товарных позиций.
- `order_item_id` также не является уникальным самостоятельно: он используется как номер позиции внутри конкретного заказа.
- Комбинация `order_id` и `order_item_id` является уникальной и не содержит пропусков, поэтому эту пару можно рассматривать как **составной `PRIMARY KEY`**.
- Полных дубликатов строк **нет**.
- Пропуски во всех столбцах **отсутствуют**.
- В таблице представлено **98 666** уникальных заказов.
- **9 803** заказа содержат более одной товарной позиции.
- Максимальное количество товарных позиций в одном заказе составляет **21**.
- `product_id` идентифицирует товар и в дальнейшем будет использоваться для связи с таблицей `products`.
- `seller_id` идентифицирует продавца и в дальнейшем будет использоваться для связи с таблицей `sellers`.
- `price` содержит стоимость товарной позиции, а `freight_value` — стоимость её доставки.
- `shipping_limit_date` содержит дату предельного срока передачи товара продавцом для отправки. При загрузке из CSV столбец имеет строковый тип (`str`), поэтому перед временным анализом его необходимо преобразовать в `datetime`.

## 4. Products

Таблица содержит информацию о характеристиках товаров, представленных в заказах.

In [22]:
# Первые строки таблицы
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [23]:
# Общая информация
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [24]:
# Проверка primary key
print("product_id уникален:", products["product_id"].is_unique)
print("Пропусков в product_id:", products["product_id"].isna().sum())

product_id уникален: True
Пропусков в product_id: 0


In [25]:
# Проверка полных дубликатов
print("Полных дубликатов:", products.duplicated().sum())

Полных дубликатов: 0


In [26]:
# Количество и доля пропусков по столбцам
missing = pd.DataFrame({
    "missing_count": products.isna().sum(),
    "missing_percent": products.isna().mean().mul(100).round(2)
})

missing

,missing_count,missing_percent
product_id,0,0.00
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


In [27]:
# Проверка структуры пропусков
product_info_missing = products[
    [
        "product_category_name",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty"
    ]
].isna().all(axis=1).sum()

physical_info_missing = products[
    [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].isna().all(axis=1).sum()

print(
    "Товаров без категории и описательных характеристик:",
    product_info_missing
)

print(
    "Товаров без физических характеристик:",
    physical_info_missing
)

Товаров без категории и описательных характеристик: 610
Товаров без физических характеристик: 2


In [28]:
# Количество категорий
print(
    "Количество категорий:",
    products["product_category_name"].nunique()
)

Количество категорий: 73


### Вывод по `products`

- Количество строк: **32 951**.
- Количество столбцов: **9**.
- Одна строка соответствует **одному товару**.
- `product_id` является уникальным идентификатором товара: значения не повторяются и не содержат пропусков. Поэтому столбец можно использовать как `PRIMARY KEY`.
- Полных дубликатов строк **нет**.
- В таблице представлено **73 категории товаров**.
- У **610 товаров (1,85%)** одновременно отсутствуют:
  - `product_category_name`;
  - `product_name_lenght`;
  - `product_description_lenght`;
  - `product_photos_qty`.
- У **2 товаров (0,01%)** одновременно отсутствуют все физические характеристики:
  - `product_weight_g`;
  - `product_length_cm`;
  - `product_height_cm`;
  - `product_width_cm`.
- `product_id` заполнен для всех товаров, поэтому наличие пропусков в характеристиках не мешает однозначно идентифицировать товар.
- Таблица содержит описательные характеристики товара — категорию, длину названия, длину описания и количество фотографий — и физические характеристики: вес, длину, высоту и ширину.
- В названиях `product_name_lenght` и `product_description_lenght` присутствует исходная опечатка `lenght`; при дальнейшем preprocessing их можно переименовать в `length`.

## 5. Sellers

Таблица содержит информацию о продавцах маркетплейса.

In [29]:
# Первые строки таблицы
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [30]:
# Общая информация
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB


In [31]:
# Проверка primary key
print("seller_id уникален:", sellers["seller_id"].is_unique)
print("Пропусков в seller_id:", sellers["seller_id"].isna().sum())

seller_id уникален: True
Пропусков в seller_id: 0


In [32]:
# Проверка полных дубликатов
print("Полных дубликатов:", sellers.duplicated().sum())

Полных дубликатов: 0


### Вывод по `sellers`

- Количество строк: **3 095**.
- Количество столбцов: **4**.
- Одна строка соответствует **одному продавцу**.
- `seller_id` является уникальным идентификатором продавца: значения не повторяются и не содержат пропусков. Поэтому столбец можно использовать как `PRIMARY KEY`.
- Полных дубликатов строк **нет**.
- Пропуски во всех столбцах **отсутствуют**.
- Таблица содержит географическую информацию о продавцах:
  - `seller_zip_code_prefix` — префикс почтового индекса;
  - `seller_city` — город;
  - `seller_state` — штат.
- `seller_zip_code_prefix` при загрузке имеет тип `int64`, однако при дальнейшем preprocessing его целесообразно рассматривать как категориальный идентификатор, а не как числовую величину.

## 6. Payments

Таблица содержит информацию об оплате заказов.

In [33]:
# Первые строки таблицы
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [34]:
# Общая информация
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


In [35]:
# Проверка идентификатора заказа
print("order_id уникален:", payments["order_id"].is_unique)
print("Пропусков в order_id:", payments["order_id"].isna().sum())

order_id уникален: False
Пропусков в order_id: 0


In [36]:
# Проверка полных дубликатов
print("Полных дубликатов:", payments.duplicated().sum())

Полных дубликатов: 0


In [37]:
# Проверка составного ключа
print(
    "Пара order_id + payment_sequential уникальна:",
    ~payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).any()
)

Пара order_id + payment_sequential уникальна: True


In [38]:
# Количество платежей на заказ
payments_per_order = payments.groupby("order_id").size()

print("Уникальных заказов:", payments["order_id"].nunique())
print(
    "Заказов с несколькими платежами:",
    (payments_per_order > 1).sum()
)
print(
    "Максимальное количество платежей на заказ:",
    payments_per_order.max()
)

Уникальных заказов: 99440
Заказов с несколькими платежами: 2961
Максимальное количество платежей на заказ: 29


In [39]:
# Способы оплаты
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [40]:
# Проверка соответствия заказов таблице payments
orders_without_payments = orders[
    ~orders["order_id"].isin(payments["order_id"])
]

print("Заказов без записи об оплате:", len(orders_without_payments))

Заказов без записи об оплате: 1


In [41]:
# Есть ли платежи для неизвестных заказов
payments_without_orders = payments[
    ~payments["order_id"].isin(orders["order_id"])
]

print(
    "Платежей для отсутствующих заказов:",
    payments_without_orders["order_id"].nunique()
)

Платежей для отсутствующих заказов: 0


In [42]:
# Диапазон количества платежей в рассрочке
print("Минимум:", payments["payment_installments"].min())
print("Максимум:", payments["payment_installments"].max())

Минимум: 0
Максимум: 24


### Вывод по `payments`

- Количество строк: **103 886**.
- Количество столбцов: **5**.
- Одна строка соответствует **отдельной записи об оплате заказа**.
- В таблице представлено **99 440 уникальных заказов**.
- `order_id` не является уникальным, поскольку одному заказу может соответствовать несколько записей об оплате.
- `payment_sequential` определяет порядковый номер платежной записи внутри конкретного заказа.
- Комбинация `order_id` и `payment_sequential` является уникальной, поэтому эту пару можно рассматривать как **составной `PRIMARY KEY`**.
- Полных дубликатов строк **нет**.
- Пропуски во всех столбцах **отсутствуют**.
- **2 961 заказ** имеет более одной записи об оплате.
- Максимальное количество записей об оплате для одного заказа составляет **29**.
- В таблице представлены следующие способы оплаты:
  - `credit_card` — 76 795;
  - `boleto` — 19 784;
  - `voucher` — 5 775;
  - `debit_card` — 1 529;
  - `not_defined` — 3.
- Наиболее распространённым способом оплаты является `credit_card`.
- Все `order_id` из таблицы `payments` присутствуют в таблице `orders`, то есть платежей для неизвестных заказов **нет**.
- При этом для **1 заказа** из таблицы `orders` отсутствует запись в `payments`. Этот случай необходимо отдельно изучить при проверке качества данных.
- `payment_installments` принимает значения от **0 до 24**.
- `payment_value` содержит сумму соответствующей платежной записи.

## 7. Reviews

Таблица содержит отзывы клиентов о заказах.

In [43]:
# Первые строки таблицы
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [44]:
# Общая информация
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [45]:
# Проверка идентификаторов
print("review_id уникален:", reviews["review_id"].is_unique)
print("Пропусков в review_id:", reviews["review_id"].isna().sum())

print("order_id уникален:", reviews["order_id"].is_unique)

review_id уникален: False
Пропусков в review_id: 0
order_id уникален: False


In [46]:
# Проверка повторяющихся review_id
review_id_counts = reviews["review_id"].value_counts()

print(
    "Повторяющихся review_id:",
    (review_id_counts > 1).sum()
)

print(
    "Максимальное число повторений одного review_id:",
    review_id_counts.max()
)

Повторяющихся review_id: 789
Максимальное число повторений одного review_id: 3


In [47]:
# Проверка составного ключа
print(
    "Пара review_id + order_id уникальна:",
    ~reviews.duplicated(
        subset=["review_id", "order_id"]
    ).any()
)

Пара review_id + order_id уникальна: True


In [48]:
# Проверка полных дубликатов
print("Полных дубликатов:", reviews.duplicated().sum())

Полных дубликатов: 0


In [49]:
# Количество и доля пропусков по столбцам
missing = pd.DataFrame({
    "missing_count": reviews.isna().sum(),
    "missing_percent": reviews.isna().mean().mul(100).round(2)
})

missing

,missing_count,missing_percent
review_id,0,0.00
order_id,0,0.00
review_score,0,0.00
review_comment_title,87656,88.34
review_comment_message,58247,58.70
review_creation_date,0,0.00
review_answer_timestamp,0,0.00


In [50]:
# Распределение оценок
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [51]:
# Диапазон оценок
print("Минимальная оценка:", reviews["review_score"].min())
print("Максимальная оценка:", reviews["review_score"].max())

Минимальная оценка: 1
Максимальная оценка: 5


In [52]:
# Количество отзывов на заказ
reviews_per_order = reviews.groupby("order_id").size()

print("Уникальных заказов:", reviews["order_id"].nunique())
print(
    "Заказов с несколькими отзывами:",
    (reviews_per_order > 1).sum()
)
print(
    "Максимальное количество отзывов на заказ:",
    reviews_per_order.max()
)

Уникальных заказов: 98673
Заказов с несколькими отзывами: 547
Максимальное количество отзывов на заказ: 3


### Вывод по `reviews`

- Количество строк: **99 224**.
- Количество столбцов: **7**.
- Одна строка соответствует **отдельной записи отзыва о заказе**.
- `review_id` не является уникальным: **789 идентификаторов отзывов** встречаются более одного раза, а максимальное число повторений одного `review_id` составляет **3**.
- `order_id` также не является уникальным, поскольку одному заказу может соответствовать несколько отзывов.
- Комбинация `review_id` и `order_id` является уникальной и не содержит пропусков, поэтому её можно рассматривать как **кандидат на составной `PRIMARY KEY` в текущем наборе данных**.
- Полных дубликатов строк **нет**.
- В таблице представлено **98 673 уникальных заказа**.
- **547 заказов** имеют более одного отзыва.
- Максимальное количество отзывов для одного заказа составляет **3**.
- `review_score` принимает значения от **1 до 5**:
  - `1` — 11 424;
  - `2` — 3 151;
  - `3` — 8 179;
  - `4` — 19 142;
  - `5` — 57 328.
- Наиболее часто встречается оценка **5**.
- Значительные пропуски присутствуют только в текстовых полях:
  - `review_comment_title` — 87 656 (**88,34%**);
  - `review_comment_message` — 58 247 (**58,70%**).
- В `review_id`, `order_id`, `review_score`, `review_creation_date` и `review_answer_timestamp` пропусков нет.
- Отсутствие заголовка или текста комментария не обязательно является ошибкой данных: пользователь мог оставить только числовую оценку.
- Столбцы `review_creation_date` и `review_answer_timestamp` при дальнейшем анализе необходимо преобразовать из строкового типа в `datetime`.

## 8. Categories

Таблица содержит соответствие названий категорий товаров на португальском языке их английским переводам.

In [53]:
# Первые строки таблицы
categories.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [54]:
# Общая информация
categories.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


In [55]:
# Проверка ключа
print(
    "product_category_name уникален:",
    categories["product_category_name"].is_unique
)
print(
    "Пропусков в product_category_name:",
    categories["product_category_name"].isna().sum()
)

product_category_name уникален: True
Пропусков в product_category_name: 0


In [56]:
# Проверка полных дубликатов
print("Полных дубликатов:", categories.duplicated().sum())

Полных дубликатов: 0


In [57]:
# Проверка уникальности английских названий категорий
print(
    "product_category_name_english уникален:",
    categories["product_category_name_english"].is_unique
)

product_category_name_english уникален: True


In [58]:
# Категории из products, для которых нет перевода
missing_category_translations = (
    products.loc[
        products["product_category_name"].notna(),
        "product_category_name"
    ]
    .loc[
        lambda s: ~s.isin(categories["product_category_name"])
    ]
    .unique()
)

print(
    "Категорий без перевода:",
    len(missing_category_translations)
)

Категорий без перевода: 2


### Вывод по `categories`

- Количество строк: **71**.
- Количество столбцов: **2**.
- Одна строка соответствует **одной паре «название категории — английский перевод»**.
- Таблица содержит соответствие названий категорий товаров на португальском языке их английским переводам.
- `product_category_name` является уникальным и не содержит пропусков, поэтому столбец можно использовать как `PRIMARY KEY`.
- `product_category_name_english` также содержит только уникальные значения.
- Полных дубликатов строк **нет**.
- Для **2 категорий**, встречающихся в таблице `products`, отсутствует соответствующий перевод в таблице `categories`.
- Поэтому при дальнейшем объединении `products` и `categories` по `product_category_name` часть товаров может остаться без английского названия категории.

## 9. Geolocation

Таблица содержит географические координаты, города и штаты, соответствующие префиксам почтовых индексов.

In [59]:
# Первые строки таблицы
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [60]:
# Общая информация
geolocation.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB


In [61]:
# Проверка идентификатора почтового индекса
print(
    "geolocation_zip_code_prefix уникален:",
    geolocation["geolocation_zip_code_prefix"].is_unique
)

print(
    "Пропусков в geolocation_zip_code_prefix:",
    geolocation["geolocation_zip_code_prefix"].isna().sum()
)

geolocation_zip_code_prefix уникален: False
Пропусков в geolocation_zip_code_prefix: 0


In [62]:
# Проверка полных дубликатов
print(
    "Полных дубликатов:",
    geolocation.duplicated().sum()
)

Полных дубликатов: 261831


In [63]:
# Повторяемость zip code prefix
geo_per_zip = geolocation[
    "geolocation_zip_code_prefix"
].value_counts()

print(
    "Уникальных zip code prefix:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)
print(
    "Максимальное количество записей на один zip code prefix:",
    geo_per_zip.max()
)

Уникальных zip code prefix: 19015
Максимальное количество записей на один zip code prefix: 1146


In [64]:
# Диапазон координат
print(
    "Latitude:",
    geolocation["geolocation_lat"].min(),
    "—",
    geolocation["geolocation_lat"].max()
)

print(
    "Longitude:",
    geolocation["geolocation_lng"].min(),
    "—",
    geolocation["geolocation_lng"].max()
)

Latitude: -36.6053744107061 — 45.06593318269697
Longitude: -101.46676644931476 — 121.10539381057764


In [65]:
# Проверка соответствия zip code prefix городам и штатам
cities_per_zip = (
    geolocation
    .groupby("geolocation_zip_code_prefix")["geolocation_city"]
    .nunique()
)

states_per_zip = (
    geolocation
    .groupby("geolocation_zip_code_prefix")["geolocation_state"]
    .nunique()
)

print(
    "Zip code prefix с несколькими городами:",
    (cities_per_zip > 1).sum()
)

print(
    "Zip code prefix с несколькими штатами:",
    (states_per_zip > 1).sum()
)

Zip code prefix с несколькими городами: 8556
Zip code prefix с несколькими штатами: 8


### Вывод по `geolocation`

- Количество строк: **1 000 163**.
- Количество столбцов: **5**.
- Одна строка соответствует **отдельной географической записи для префикса почтового индекса**. Одному префиксу может соответствовать несколько строк.
- Таблица содержит географическую информацию для префиксов почтовых индексов: координаты, город и штат.
- `geolocation_zip_code_prefix` не является уникальным: в таблице содержится **19 015 уникальных префиксов почтовых индексов**.
- Один `geolocation_zip_code_prefix` может соответствовать нескольким географическим записям. Максимальное количество записей для одного префикса составляет **1 146**.
- Пропуски во всех столбцах **отсутствуют**.
- В таблице обнаружено **261 831 полностью дублирующихся строк**. При дальнейшем preprocessing необходимо определить, можно ли безопасно удалить такие повторяющиеся записи.
- Для **8 556 префиксов почтовых индексов** встречается более одного значения города, а для **8 префиксов** — более одного штата. Поэтому перед приведением таблицы к одной записи на `geolocation_zip_code_prefix` необходимо определить способ выбора географических данных.
- В исходной таблице отсутствует простой естественный `PRIMARY KEY`, поскольку один префикс почтового индекса может встречаться во множестве строк.
- Координаты находятся в диапазонах:
  - `geolocation_lat`: от **−36,61** до **45,07**;
  - `geolocation_lng`: от **−101,47** до **121,11**.
- В координатах присутствуют крайние значения, которые стоит отдельно проверить на выбросы перед географическим анализом.
- `geolocation_zip_code_prefix` имеет тип `int64`, однако префикс почтового индекса является идентификатором, а не числовой величиной, поэтому при preprocessing его целесообразно преобразовать в строковый тип.
- Для дальнейшего объединения с таблицами клиентов и продавцов потребуется привести `geolocation` к более удобной гранулярности, например к одной записи на `geolocation_zip_code_prefix`.

## 10. Relationships

Проверим связи между исходными таблицами и наличие внешних ключей,
для которых отсутствуют соответствующие записи в связанных таблицах.

In [66]:
def check_relationship(child, child_key, parent, parent_key):
    child_values = child[child_key].dropna()
    parent_values = parent[parent_key].dropna()

    unmatched_child = child_values[
        ~child_values.isin(parent_values)
    ]

    parent_without_child = parent_values[
        ~parent_values.isin(child_values)
    ]

    return {
        "child_rows": len(child),
        "unique_child_keys": child_values.nunique(),
        "unmatched_child_keys": unmatched_child.nunique(),
        "parent_keys_without_child": parent_without_child.nunique()
    }

In [67]:
relationships = {
    "orders.customer_id → customers.customer_id": check_relationship(
        orders,
        "customer_id",
        customers,
        "customer_id"
    ),
    "order_items.order_id → orders.order_id": check_relationship(
        order_items,
        "order_id",
        orders,
        "order_id"
    ),
    "order_items.product_id → products.product_id": check_relationship(
        order_items,
        "product_id",
        products,
        "product_id"
    ),
    "order_items.seller_id → sellers.seller_id": check_relationship(
        order_items,
        "seller_id",
        sellers,
        "seller_id"
    ),
    "payments.order_id → orders.order_id": check_relationship(
        payments,
        "order_id",
        orders,
        "order_id"
    ),
    "reviews.order_id → orders.order_id": check_relationship(
        reviews,
        "order_id",
        orders,
        "order_id"
    )
}

pd.DataFrame(relationships).T

,child_rows,unique_child_keys,unmatched_child_keys,parent_keys_without_child
orders.customer_id → customers.customer_id,99441,99441,0,0
order_items.order_id → orders.order_id,112650,98666,0,775
order_items.product_id → products.product_id,112650,32951,0,0
order_items.seller_id → sellers.seller_id,112650,3095,0,0
payments.order_id → orders.order_id,103886,99440,0,1
reviews.order_id → orders.order_id,99224,98673,0,768


In [68]:
product_categories = (
    products["product_category_name"]
    .dropna()
)

print(
    "Категорий товаров без перевода:",
    product_categories[
        ~product_categories.isin(
            categories["product_category_name"]
        )
    ].nunique()
)

Категорий товаров без перевода: 2


In [69]:
print(
    "Префиксов клиентов без geolocation:",
    customers.loc[
        ~customers["customer_zip_code_prefix"].isin(
            geolocation["geolocation_zip_code_prefix"]
        ),
        "customer_zip_code_prefix"
    ].nunique()
)

print(
    "Префиксов продавцов без geolocation:",
    sellers.loc[
        ~sellers["seller_zip_code_prefix"].isin(
            geolocation["geolocation_zip_code_prefix"]
        ),
        "seller_zip_code_prefix"
    ].nunique()
)

Префиксов клиентов без geolocation: 157
Префиксов продавцов без geolocation: 7


### Вывод по связям между таблицами

- Все проверенные внешние ключи ссылаются на существующие записи: проблем с `customer_id`, `order_id`, `product_id` и `seller_id` не найдено.
- `customers` и `orders` связаны по `customer_id` как **1:1**. При этом один покупатель по `customer_unique_id` может сделать несколько заказов.
- Связь `orders → order_items` имеет тип **1:0..M**: заказ может содержать несколько позиций, но у **775 заказов** записей в `order_items` нет.
- Связи `products → order_items` и `sellers → order_items` имеют тип **1:M**: один товар или продавец может встречаться во многих позициях заказов.
- Связь `orders → payments` имеет тип **1:0..M**. Для **1 заказа** запись об оплате отсутствует.
- Связь `orders → reviews` имеет тип **1:0..M**. Для **768 заказов** отзывов нет.
- Для **2 категорий товаров** нет английского перевода в `categories`.
- Для **157 почтовых префиксов клиентов** и **7 префиксов продавцов** нет записей в `geolocation`.
- В `geolocation` один `geolocation_zip_code_prefix` может встречаться несколько раз, поэтому перед объединением с `customers` и `sellers` таблицу нужно привести к одной записи на почтовый префикс.
- После этого для объединения с `geolocation` лучше использовать `LEFT JOIN`, чтобы не потерять клиентов или продавцов без географических данных.

В целом таблицы хорошо связываются между собой. Основные особенности, которые нужно учесть дальше, — заказы без позиций, платежей или отзывов, отсутствующие переводы категорий и неполные данные в `geolocation`.